# Lab 7: Nuclear Magnetic Resonance (NMR) Spectroscopy

---

## Purpose

In this lab you will:

**Chemistry:**
- Understand how nuclear spin interacts with magnetic fields to produce energy splitting
- Connect Larmor precession frequency to resonance conditions
- Relate T1 and T2 relaxation times to peak intensity and linewidth

**Coding:**
- Apply Fourier transforms to convert time-domain signals to frequency-domain spectra
- Write functions to calculate NMR parameters from fundamental constants
- Use interactive visualizations to explore parameter dependencies

**Real-world connection:** NMR is one of the most powerful techniques for structure determination in chemistry. Understanding the physics behind NMR helps interpret spectra and troubleshoot experiments.

---

## Estimated Time: 75-90 minutes

---

## Success Criteria

By the end of this lab, you should be able to:
- [ ] Calculate Larmor frequency for a given nucleus and field strength
- [ ] Convert between frequency and chemical shift in ppm
- [ ] Predict how T2 affects peak linewidth
- [ ] Transform an FID signal to a frequency-domain spectrum using FFT
- [ ] Explain the connection between quantum-level precession and macroscopic NMR signals

---

# Libraries

Run this cell before proceeding.

In [ ]:
# Standard scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fourier transforms
from scipy.fft import fft, ifft, fftfreq

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Helper functions for this lab
from nmr_helper import (
    plot_precession, plot_fid, plot_nmr_visualization,
    larmor_frequency, larmor_frequency_hz, chemical_shift_ppm,
    fwhm_from_T2, T2_from_fwhm, generate_fid, fid_to_spectrum,
    GAMMA_H, GAMMA_C13, HBAR
)

---

# Part 1: Signal Processing Warmup (15 min)

Before diving into NMR physics, let's practice the signal processing technique that makes modern NMR possible: **Fourier transformation**.

In NMR, we don't directly measure resonance frequencies. Instead, we:
1. Excite the sample with an RF pulse
2. Record the time-domain signal (Free Induction Decay, or FID)
3. Convert to frequency domain using FFT to get the spectrum

This warmup simulates step 3.

## Given Code

Run this cell to generate a composite signal with three frequencies plus noise.

In [ ]:
# ============================================
# SUBGOAL: Generate a test signal
# ============================================

# Time domain settings
sampling_rate = 500  # Hz
duration = 1.0       # seconds
t = np.linspace(0, duration, int(sampling_rate * duration))

# Generate a composite signal (3 frequencies + noise)
freqs = [5, 15, 50]  # Hz
amplitudes = [1.0, 0.5, 0.25]

# Generate the signal
signal = np.zeros_like(t)
for freq, amp in zip(freqs, amplitudes):
    signal += amp * np.sin(2 * np.pi * freq * t)

# Add noise
noise_level = 0.3
noise = noise_level * np.random.randn(len(t))
signal = signal + noise

print(f"Signal generated with {len(signal)} points")
print(f"Frequencies: {freqs} Hz with amplitudes {amplitudes}")

## Your Turn: Signal Processing Pipeline

Complete the following four cells to process the signal.

**Goal:** Transform a noisy time-domain signal to frequency domain, apply filtering, and transform back.

**Hint:** The FFT functions are already imported: `fft()`, `ifft()`, `fftfreq()`

In [ ]:
# ============================================
# SUBGOAL: Plot original signal in time domain
# ============================================

# TODO: Create a plot of signal vs t
# Include: title, xlabel ('Time (s)'), ylabel ('Amplitude'), and grid

# YOUR CODE HERE


In [ ]:
# ============================================
# SUBGOAL: Compute and plot the FFT
# ============================================

# Step 1: Get the number of points
N = len(t)

# Step 2: Compute FFT of the signal
# TODO: Replace None with the correct function call
fft_signal = None  # Hint: fft(signal)

# Step 3: Get frequency array
freqs_array = fftfreq(N, 1/sampling_rate)

# Step 4: Get positive frequencies only (for plotting)
pos_freqs = freqs_array[:N//2]
pos_fft = np.abs(fft_signal[:N//2]) / N  # Normalize

# TODO: Plot the frequency spectrum
# Plot pos_fft vs pos_freqs
# Set xlim to (0, 100) to focus on relevant frequencies
# Include: title, xlabel ('Frequency (Hz)'), ylabel ('Amplitude'), and grid

# YOUR CODE HERE


In [ ]:
# ============================================
# SUBGOAL: Apply low-pass filter
# ============================================

# TODO: Set a cutoff frequency to remove the 50 Hz component but keep 5 and 15 Hz
cutoff_freq = None  # Choose a value between 15 and 50 Hz

# Create filtered copy
filtered_fft = fft_signal.copy()

# Zero out frequencies above cutoff (both positive and negative)
filter_mask = np.abs(freqs_array) > cutoff_freq
filtered_fft[filter_mask] = 0

# Plot filtered spectrum
pos_filtered_fft = np.abs(filtered_fft[:N//2]) / N

# TODO: Plot the filtered frequency spectrum
# Same axes as before, but add axvspan to highlight filtered region
# Example: plt.axvspan(cutoff_freq, 100, alpha=0.2, color='red')

# YOUR CODE HERE


In [ ]:
# ============================================
# SUBGOAL: Convert back to time domain
# ============================================

# TODO: Apply inverse FFT to get filtered signal
# Hint: ifft() returns complex values - use .real to get real part
filtered_signal = None  # Replace with ifft(filtered_fft).real

# TODO: Plot the filtered signal in time domain
# Compare to original - the 50 Hz component and noise should be reduced

# YOUR CODE HERE


### Short Response Questions

**Q1.1 (3 pts):** Looking at your frequency-domain plot, how do the peaks relate to the frequencies used to generate the signal? What do the peak heights tell you?

**Q1.2 (3 pts):** After applying the low-pass filter and transforming back, how does the filtered signal compare to the original? What information was lost? What was preserved?

**A1.1:**

*Your answer here*

**A1.2:**

*Your answer here*

---

# Part 2: Magnetic Energy Splitting (15 min)

Now let's connect to NMR physics. Nuclei with spin-1/2 (like ¹H) have two energy states in a magnetic field.

## Key Equations

**Energy splitting:**
$$\Delta E = \gamma \hbar B_{eff}$$

where $B_{eff} = (1-\sigma)B_0$ accounts for shielding.

**Larmor frequency** (precession/resonance frequency):
$$\omega_0 = \gamma B_{eff} = \frac{\Delta E}{\hbar}$$

**Chemical shift** (in ppm):
$$\delta = \frac{\nu_{sample} - \nu_{reference}}{\nu_{spectrometer}} \times 10^6$$

## Task 2.1: Calculate Larmor Frequency

Write a function to calculate the Larmor frequency in MHz for a proton in a given magnetic field.

**Given constants (from nmr_helper):**
- `GAMMA_H = 2.6752e8` rad/(s·T) - gyromagnetic ratio for ¹H
- `GAMMA_C13 = 6.7282e7` rad/(s·T) - gyromagnetic ratio for ¹³C

In [ ]:
def calculate_larmor_mhz(gamma, B0):
    """
    Calculate the Larmor frequency in MHz.
    
    Parameters
    ----------
    gamma : float
        Gyromagnetic ratio in rad/(s*T)
    B0 : float
        Magnetic field strength in Tesla
    
    Returns
    -------
    float
        Larmor frequency in MHz
    """
    # TODO: Calculate omega_0 = gamma * B0 (in rad/s)
    # TODO: Convert to Hz by dividing by 2*pi
    # TODO: Convert to MHz by dividing by 1e6
    
    # YOUR CODE HERE
    return None  # Replace with your calculation

In [ ]:
# Test your function
# A 500 MHz spectrometer uses B0 ≈ 11.74 T

B0_test = 11.74  # Tesla
freq_H = calculate_larmor_mhz(GAMMA_H, B0_test)
freq_C = calculate_larmor_mhz(GAMMA_C13, B0_test)

print(f"At B0 = {B0_test} T:")
print(f"  ¹H Larmor frequency: {freq_H:.1f} MHz")
print(f"  ¹³C Larmor frequency: {freq_C:.1f} MHz")

# Check: ¹H should be ~500 MHz, ¹³C should be ~125 MHz
if freq_H is not None:
    assert 495 < freq_H < 505, f"Expected ~500 MHz for ¹H, got {freq_H:.1f}"
    print("\n✓ Your calculation is correct!")

## Task 2.2: Calculate Chemical Shift

A proton in a sample resonates at 500,000,350 Hz. The TMS reference resonates at exactly 500,000,000 Hz. Calculate the chemical shift in ppm.

In [ ]:
# Given values
nu_sample = 500_000_350  # Hz (sample frequency)
nu_reference = 500_000_000  # Hz (TMS reference)
nu_spectrometer = 500_000_000  # Hz (spectrometer frequency)

# TODO: Calculate chemical shift using the formula:
# delta = (nu_sample - nu_reference) / nu_spectrometer * 1e6

delta = None  # YOUR CODE HERE

print(f"Chemical shift: {delta} ppm")

## Task 2.3: Interactive Energy Exploration

Create a function that calculates the energy splitting and an interactive plot to explore how it depends on field strength.

In [ ]:
def delta_E(gamma, sigma, B0):
    """
    Calculate the energy splitting between spin states.
    
    Parameters
    ----------
    gamma : float
        Gyromagnetic ratio in rad/(s*T)
    sigma : float
        Shielding constant (dimensionless, typically ~1e-6)
    B0 : float
        Applied magnetic field in Tesla
    
    Returns
    -------
    float
        Energy splitting in Joules
    """
    # TODO: Calculate B_eff = (1 - sigma) * B0
    # TODO: Calculate Delta_E = gamma * hbar * B_eff
    # Hint: HBAR is already imported
    
    # YOUR CODE HERE
    return None  # Replace with your calculation

In [ ]:
def plot_energy_splitting(B0=11.74):
    """
    Plot energy splitting as a function of shielding constant.
    """
    # Range of shielding constants (in ppm, converted to dimensionless)
    sigma_ppm = np.linspace(0, 12, 100)
    sigma = sigma_ppm * 1e-6  # Convert to dimensionless
    
    # Calculate energy for each sigma
    energies = [delta_E(GAMMA_H, s, B0) for s in sigma]
    
    # Convert to more useful units (meV or μeV)
    energies_uev = [e / 1.602e-19 * 1e6 if e is not None else 0 for e in energies]
    
    plt.figure(figsize=(8, 5))
    plt.plot(sigma_ppm, energies_uev)
    plt.xlabel('Shielding constant σ (ppm)')
    plt.ylabel('Energy splitting (μeV)')
    plt.title(f'NMR Energy Splitting at B₀ = {B0} T')
    plt.grid(True, alpha=0.3)
    plt.show()

# TODO: Use widgets.interact() to make this interactive
# B0 should range from 1 to 20 Tesla

# YOUR CODE HERE


### Short Response Questions

**Q2.1 (4 pts):** Why is the ¹H Larmor frequency about 4× higher than ¹³C at the same field strength? What property of the nucleus determines this?

**Q2.2 (4 pts):** A proton near an electronegative atom (like oxygen) experiences less shielding than a proton on a methyl group. Based on your calculations, which proton would appear at higher ppm in the NMR spectrum? Explain your reasoning.

**A2.1:**

*Your answer here*

**A2.2:**

*Your answer here*

---

# Part 3: Magnetic Precession (10 min)

When a nuclear magnetic moment is not aligned with the magnetic field, it **precesses** - rotating around the field axis at the Larmor frequency.

The expectation values of the spin components oscillate:
$$\langle I_x \rangle = \frac{\hbar}{2}\sin(\alpha)\cos(\omega_0 t)$$
$$\langle I_y \rangle = -\frac{\hbar}{2}\sin(\alpha)\sin(\omega_0 t)$$

where $\alpha$ is the angle from the z-axis and $\omega_0 = \gamma B_0$ is the Larmor frequency.

## Explore: Spin Precession

Use the interactive visualization to explore how angle, field strength, and time affect precession.

In [ ]:
# TODO: Use widgets.interact() to create an interactive plot
# Use plot_precession(angle, B0, time) from nmr_helper
#
# Parameters:
#   angle: 0 to 180 degrees
#   B0: 0.2 to 2.0
#   time: 0 to 1

# YOUR CODE HERE


### Short Response Questions

**Q3.1 (4 pts):** At what angle(s) does precession NOT occur? At all other angles, does the **frequency** of precession depend on the angle? Explain physically why this makes sense.

**Q3.2 (3 pts):** How does increasing B0 affect the precession? What does this tell you about how spectrometer field strength affects measurement?

**A3.1:**

*Your answer here*

**A3.2:**

*Your answer here*

---

# Part 4: Free Induction Decay and NMR Signals (15 min)

Individual nuclear spins are invisible - we observe the **macroscopic magnetization** $\mathbf{M}$, the sum of all nuclear magnetic moments.

After a 90° RF pulse tips $\mathbf{M}$ into the xy-plane, it precesses and induces a voltage in the detection coil. This **Free Induction Decay (FID)** is our raw NMR signal:

$$FID(t) \propto e^{-t/T_2} \sin(\omega_0 t)$$

The Fourier transform of the FID gives us the NMR spectrum!

## Explore: Magnetization and FID

Use the interactive visualization to see how precessing magnetization generates the FID signal.

In [ ]:
# TODO: Use widgets.interact() to create an interactive plot
# Use plot_fid(B0, decay_rate, time) from nmr_helper
#
# Parameters:
#   B0: 0.2 to 2.0
#   decay_rate: 0.2 to 2.0  
#   time: (0, 3, 0.05)  # Use exactly this tuple for the time slider

# YOUR CODE HERE


## Task 4.1: FID to Spectrum

Generate a synthetic FID with multiple frequency components and transform it to a spectrum.

In [ ]:
# ============================================
# SUBGOAL: Generate a multi-peak FID
# ============================================

# Use generate_fid from nmr_helper
# This simulates an NMR sample with multiple chemically distinct nuclei

# Parameters for three peaks
peak_frequencies = [50, 120, 200]  # Hz (offset from reference)
peak_amplitudes = [1.0, 0.6, 0.3]  # Relative peak heights
peak_T2 = [0.5, 0.3, 0.8]  # T2 values in seconds

# Generate FID
t_fid, fid = generate_fid(peak_frequencies, peak_amplitudes, peak_T2, 
                          t_max=2.0, sampling_rate=1000)

# Plot the FID (real part only)
plt.figure(figsize=(10, 4))
plt.plot(t_fid, fid.real, 'b-', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Signal')
plt.title('Simulated Free Induction Decay')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================
# SUBGOAL: Transform FID to spectrum
# ============================================

# TODO: Use fid_to_spectrum() from nmr_helper to get frequency and spectrum
# freq, spectrum = fid_to_spectrum(t_fid, fid)

freq, spectrum = None, None  # YOUR CODE HERE

# TODO: Plot the spectrum
# - Set xlim to (0, 300) to focus on relevant frequencies
# - Label axes appropriately
# - Mark the expected peak positions

# YOUR CODE HERE


## Task 4.2: Predict Linewidth from T2

The Full Width at Half Maximum (FWHM) of an NMR peak is related to T2:

$$FWHM = \frac{1}{\pi T_2}$$

Use the `fwhm_from_T2()` function to predict linewidths, then verify by examining your spectrum.

In [ ]:
# Calculate predicted linewidths
print("Predicted linewidths:")
for i, T2 in enumerate(peak_T2):
    # TODO: Use fwhm_from_T2() to calculate the expected linewidth
    fwhm = None  # YOUR CODE HERE
    print(f"  Peak at {peak_frequencies[i]} Hz: T2 = {T2} s → FWHM = {fwhm} Hz")

### Short Response Questions

**Q4.1 (4 pts):** How does macroscopic magnetization behave similarly to individual spin magnetic moments? How is it different?

**Q4.2 (4 pts):** Looking at your spectrum, which peak is the sharpest? Which has the longest T2? Explain the physical relationship between T2 and linewidth using the uncertainty principle.

**A4.1:**

*Your answer here*

**A4.2:**

*Your answer here*

---

# Part 5: T1 and T2 Relaxation (10 min)

Two relaxation processes determine peak appearance:

| Relaxation | Symbol | Affects | Physical Origin |
|------------|--------|---------|----------------|
| Transverse (spin-spin) | T2 | Peak width | Loss of phase coherence |
| Longitudinal (spin-lattice) | T1 | Peak intensity | Energy exchange with surroundings |

**Key relationships:**
- Short T2 → Fast decay → Broad peaks
- Long T2 → Slow decay → Sharp peaks
- T1 affects signal intensity in multi-pulse experiments

## Explore: NMR Spectrum Parameters

Use the interactive visualization to see how shielding, T1, and T2 affect the FID and spectrum.

In [ ]:
# TODO: Use widgets.interact() to create an interactive plot
# Use plot_nmr_visualization(sigma, T_2, T_1) from nmr_helper
#
# Parameters:
#   sigma: 0 to 15 (ppm scale)
#   T_2: 0.1 to 2.0
#   T_1: 0.1 to 2.0

# YOUR CODE HERE


### Short Response Questions

**Q5.1 (4 pts):** Which parameter (σ, T1, or T2) controls each of the following features?
- Peak position (chemical shift)
- Peak width
- Peak height (in repeated scans)

**Q5.2 (4 pts):** Why does a larger T2 result in sharper peaks? Explain using both the FID signal and the uncertainty principle.

**A5.1:**

*Your answer here*

**A5.2:**

*Your answer here*

---

# Part 6: Putting It Together - Solve from Scratch (10 min)

Apply what you've learned to solve a complete NMR problem.

## Problem

You are analyzing a proton NMR spectrum acquired on a 600 MHz spectrometer (B0 = 14.1 T).

A peak appears at 7.26 ppm with a linewidth of 2.0 Hz.

Calculate:
1. The absolute resonance frequency of this peak (in Hz)
2. The T2 relaxation time for this nucleus
3. The energy splitting between spin states (in J and in eV)

In [ ]:
# Given values
spectrometer_freq = 600e6  # Hz (600 MHz)
B0 = 14.1  # Tesla
delta_ppm = 7.26  # ppm
linewidth = 2.0  # Hz

# ============================================
# 1. Calculate absolute frequency
# ============================================
# The peak is delta_ppm downfield from TMS
# nu_sample = nu_TMS + (delta_ppm / 1e6) * nu_spectrometer

nu_tms = spectrometer_freq  # TMS is at spectrometer frequency by definition
nu_sample = None  # YOUR CODE HERE

print(f"1. Absolute frequency: {nu_sample} Hz")
if nu_sample:
    print(f"   (That's {nu_sample/1e6:.6f} MHz)")

# ============================================
# 2. Calculate T2 from linewidth
# ============================================
# Use T2_from_fwhm() or calculate directly: T2 = 1/(pi * FWHM)

T2 = None  # YOUR CODE HERE

print(f"\n2. T2 relaxation time: {T2} s")
if T2:
    print(f"   (That's {T2*1000:.1f} ms)")

# ============================================
# 3. Calculate energy splitting
# ============================================
# Delta_E = hbar * omega = hbar * gamma * B0
# Note: The chemical shift correction is tiny, so we can ignore it for energy

delta_E_joules = None  # YOUR CODE HERE
delta_E_eV = None  # Convert: 1 eV = 1.602e-19 J

print(f"\n3. Energy splitting:")
print(f"   {delta_E_joules} J")
print(f"   {delta_E_eV} eV")

---

# Reflection

Answer these synthesis questions to demonstrate your understanding of NMR spectroscopy.

### Short Response Questions

**Q7.1 (5 pts):** What physical properties does NMR measure, and what information does this provide about molecular structure? Give specific examples.

**Q7.2 (5 pts):** How is NMR connected to statistical mechanics? Describe both:
- The quantum-level behavior (individual nuclear spins)
- The macroscopic, statistical behavior (bulk magnetization)

How do these two perspectives connect to produce the experimental signal?

**A7.1:**

*Your answer here*

**A7.2:**

*Your answer here*

---

# References

1. Levine, I. N. *Physical Chemistry*, 6th ed.; McGraw-Hill: New York, 2009; Chapter 15.
2. Keeler, J. *Understanding NMR Spectroscopy*, 2nd ed.; Wiley: Chichester, 2010.
3. Friebolin, H. *Basic One- and Two-Dimensional NMR Spectroscopy*, 5th ed.; Wiley-VCH: Weinheim, 2011.